# EDA on a GLTF 3D Model File

The [GLTF](../../hosting-3d-model/analysis_threejs.md) is an open source file format that allows for efficient rendering on the web. In this notebook, I explore the basics of what this file structure looks like and see if any optimizations can be made. We do have an ulterioir agenda for this notebook, and that is to incorporate our [mesh decimation script](../../reducing-mesh-density/notebooks/mesh-simplification.ipynb) within an existing GLTF scene, and hopefully optimize our geometry such that it can be used in GTLF format. 

In [1]:
import pandas as pd
import numpy as np

import pygltflib
import struct

The GLTF file follows a JSON-style format, meaning that each element is a child of another one. The main library we shall be using for this task is the `pygltflib` library. This includes a host of tools for reading and writing GLTF and GLB files.

In [2]:
import trimesh

Another library we shall be exploring is the `trimesh` library. [Trimesh](https://trimesh.org/) appears to be more widely used, altough pygltflib seems to have the functionality we require.

Let's start with pygltflib. We work with this basic script as found in the [docs](https://pypi.org/project/pygltflib/). Here we load a simple gltf file to memory.

In [3]:
filename = "../../../models/foot/human-foot-hires.glb"
gltf = pygltflib.GLTF2().load(filename)

Let's access the first node in the scene.

In [4]:
current_scene = gltf.scenes[gltf.scene]
node_index = current_scene.nodes[0]

node_index

0

Let's get the first mesh in the scene.

In [5]:
mesh = gltf.meshes[node_index]
mesh

Mesh(extensions={}, extras={}, primitives=[Primitive(extensions={}, extras={}, attributes=Attributes(POSITION=0, NORMAL=1, TANGENT=None, TEXCOORD_0=2, TEXCOORD_1=None, COLOR_0=None, JOINTS_0=None, WEIGHTS_0=None), indices=3, mode=4, material=0, targets=[])], weights=[], name='Cube;hires')

The next code block from the docs exports the vertices from this mesh.

In [6]:
for primitive in mesh.primitives:

    # get the binary data for this mesh primitive from the buffer
    accessor = gltf.accessors[primitive.attributes.POSITION]
    bufferView = gltf.bufferViews[accessor.bufferView]
    buffer = gltf.buffers[bufferView.buffer]
    data = gltf.get_data_from_buffer_uri(buffer.uri)

    # pull each vertex from the binary buffer and convert it into a tuple of python floats
    vertices = []
    for i in range(accessor.count):
        index = bufferView.byteOffset + accessor.byteOffset + i*12  # the location in the buffer of this vertex
        d = data[index:index+12]  # the vertex data
        v = struct.unpack("<fff", d)   # convert from base64 to three floats
        vertices.append(v)
        print(i, v)

0 (0.059607651084661484, 0.38341936469078064, -0.04792476072907448)
1 (-0.030598677694797516, -0.01601026952266693, -0.02000928483903408)
2 (0.05239549279212952, 0.4023452699184418, 0.10257382690906525)
3 (0.04939287155866623, -0.015810955315828323, -0.022422943264245987)
4 (-0.061567358672618866, 0.42185527086257935, -0.06672687083482742)
5 (-0.09298700094223022, 0.022494051605463028, 0.09659532457590103)
6 (-0.09703325480222702, 0.45046985149383545, 0.15233224630355835)
7 (0.11790001392364502, 0.008622212335467339, -0.008328386582434177)
8 (-0.030263425782322884, 0.007944571785628796, 0.07137668877840042)
9 (0.04509124532341957, -0.00760089373216033, 0.06834688782691956)
10 (-0.005898107774555683, 0.4344777464866638, 0.1600494384765625)
11 (-0.0572814866900444, 0.45368483662605286, 0.17707258462905884)
12 (-0.031144101172685623, -0.012257005088031292, -0.11187183856964111)
13 (0.03342190384864807, 0.0005386420525610447, 0.14107629656791687)
14 (-0.015178482048213482, 0.39298489689826

And here we have the vertices of our human foot model.

It's important to understand how this code works since we will need to tweak it very heavily to get it to work for our use case. For that, we must understand how the GLTF file format stores and retrieves data. From the official [Khronos website](https://www.khronos.org/gltf/), we found this [handly flowchart](../../../gltf20-reference-guide.pdf) that describes the hierarchy of this file. Here is a brief screenshot.

![GLTF file format hierarchy](../img/gltf-file-hierarchy.png)

A lot of the items in this list are containers for data which we wont need. For example, our scene won't have any animation, so there's no need to understand how this works. For now, we will be looking at the following.

- `scene`
- `node`
- `mesh`
- `accessor`
- `bufferView`
- `buffer`

The GLTF file format uses indexing to maintain order in the structure of the file. The data is usually stored as a linear array (i.e. data is encoded in a 1D incremental structure). Each one of these data structures above save indices of the specific section of the array that contains theri associated information.

To drive this point home, we open a `.gltf` file in a text editor.


> A side note: there are 2 main types of file format that fall under the `gltf` file umbrella- .gltf files and .glb files. gltf stores the indices of our array in human readable text, and as a JSON format, while glb (also called gltf binary), saves the data as raw bytes. This means the glb file is better compressed, but loses the human readable aspect. We shall work with gltf files for demo pruposes, but glb in implementation.

Here is what the structure of the `piperacks_valve_only_decimate.gltf` file looks like in VS Code.

![GLTF file in a text editor](../img/piperacks-valve-only-gltf-format-text.png)

Here, we observe the hierarchy of the file format. The tree starts at the root node, which contains the children: "asset", "scene", "scenes", "nodes", "meshes", and "accessors". Each one of these sections contains indices which refer to the section below it.

For example, the `scene` node contains the index 0 -> this refers to the 0-th index of the `scenes` branch. Here, since we only have one scene in our file, the default index for this is 0. Similarly, the scenes branch refers to the `nodes` branch, index 0, which then refers to the meshes branch, also index 0. Each item index is referenced at the different levels of the tree. If we had multiple meshes in our scene, these would get the indices 1, 2, 3 and so on. 

The second half of this file contains further subdivisions of the tree-> "accessors", "bufferViews" and "buffers". This is where the bulk of the data is stored and referenced.

![GLTF file in a text editor -2](../img/piperacks-valve-only-gltf-format-text_2.png)

Accessors store information on the POSITION, ROTATION and SCALE of the data. This is saved in the different `bufferViews`, which each reference different parts of the `buffer`. Finally, the Buffer also includes `bytelength` and `byteoffset` data, which referes to our linear arrar concept mentioned earlier. Essentially, provides information on which index to start counting at in the array, and how long to continue for.

For example, the first `accessor` here refers to `bufferview` 0. BufferView index 0 (the first one in the list), references buffer 0 (the first and only one in the list). The buffer refers to a `uri`, which is a reference to where the actual data is stored, usually in binary format.

With this brief overview out of the way, let's proceed with EDA on our mesh.

We were able to extract vertex data from our foot mesh. What about triangles? Once again, from the docs, we see this method.

In [7]:
binary_blob = gltf.binary_blob()

triangles_accessor = gltf.accessors[gltf.meshes[0].primitives[0].indices]
triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
triangles = np.frombuffer(
    binary_blob[
        triangles_buffer_view.byteOffset
        + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
        + triangles_buffer_view.byteLength
    ],
    dtype="uint8",
    count=triangles_accessor.count,
).reshape((-1, 3))

triangles

array([[224,   2,  31],
       [  0,  33,   0],
       [224,   2,  33],
       ...,
       [  1, 172,   1],
       [159,   1, 172],
       [  1, 164,   1]], shape=(1586, 3), dtype=uint8)

Seems to be working fine. We extend this same knowledge to acquire our points array as well -->

In [8]:
points_accessor = gltf.accessors[gltf.meshes[0].primitives[0].attributes.POSITION]
points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
points = np.frombuffer(
    binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype="float32",
    count=points_accessor.count * 3,
).reshape((-1, 3))

points

array([[ 0.05960765,  0.38341936, -0.04792476],
       [-0.03059868, -0.01601027, -0.02000928],
       [ 0.05239549,  0.40234527,  0.10257383],
       ...,
       [ 0.17421806,  0.03086693,  0.4610686 ],
       [ 0.16315834,  0.03547674,  0.39787173],
       [-0.05680574,  0.01286671,  0.5708645 ]],
      shape=(822, 3), dtype=float32)

Good. We now have the same data in the same format that was used in our mesh decimation algorithm [EDA notebook](../../reducing-mesh-density/notebooks/mesh-simplification.ipynb). As a reminder, in that notebook, applied a mesh decimation script on an object, by preserving the vertex structure of the original. This way, the new mesh is created using the vertex indices of the old one, unlocking LOD capabilities, as well as saving memory.

Before we can explore this decimation, I'd like to see how we can recreate a gltf file from the given 2 face and vertex matrices above. This will be the output from MeshLab, so will be useful to know. Once again, we get this information from the pyltflib API.

In [9]:
points = np.array(
    [
        [-0.5, -0.5, 0.5],
        [0.5, -0.5, 0.5],
        [-0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5],
        [0.5, -0.5, -0.5],
        [-0.5, -0.5, -0.5],
        [0.5, 0.5, -0.5],
        [-0.5, 0.5, -0.5],
    ],
    dtype="float32",
)
triangles = np.array(
    [
        [0, 1, 2],
        [3, 2, 1],
        [1, 0, 4],
        [5, 4, 0],
        [3, 1, 6],
        [4, 6, 1],
        [2, 3, 7],
        [6, 7, 3],
        [0, 2, 5],
        [7, 5, 2],
        [5, 7, 4],
        [6, 4, 7],
    ],
    dtype="uint8",
)

In [10]:
triangles_binary_blob = triangles.flatten().tobytes()
points_binary_blob = points.tobytes()
gltf = pygltflib.GLTF2(
    scene=0,
    scenes=[pygltflib.Scene(nodes=[0])],
    nodes=[pygltflib.Node(mesh=0)],
    meshes=[
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=1), indices=0
                )
            ]
        )
    ],
    accessors=[
        pygltflib.Accessor(
            bufferView=0,
            componentType=pygltflib.UNSIGNED_BYTE,
            count=triangles.size,
            type=pygltflib.SCALAR,
            max=[int(triangles.max())],
            min=[int(triangles.min())],
        ),
        pygltflib.Accessor(
            bufferView=1,
            componentType=pygltflib.FLOAT,
            count=len(points),
            type=pygltflib.VEC3,
            max=points.max(axis=0).tolist(),
            min=points.min(axis=0).tolist(),
        ),
    ],
    bufferViews=[
        pygltflib.BufferView(
            buffer=0,
            byteLength=len(triangles_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER,
        ),
        pygltflib.BufferView(
            buffer=0,
            byteOffset=len(triangles_binary_blob),
            byteLength=len(points_binary_blob),
            target=pygltflib.ARRAY_BUFFER,
        ),
    ],
    buffers=[
        pygltflib.Buffer(
            byteLength=len(triangles_binary_blob) + len(points_binary_blob)
        )
    ],
)
gltf.set_binary_blob(triangles_binary_blob + points_binary_blob)

In [11]:
filename2 = "test.glb"
gltf.save(filename2)

True

Opening this exported model in blender show the following ->

![Test GLB file in Blender](../img/test-glb-in-blender.png)

Looks like a simple cube, but the implications are crazy. We just saved 2 raw arrays of vertices and faces to glb file.

## Adding LODs With Shared Arrays

We have already created Array versions of our foot mesh in the [Decimation EDA](../../reducing-mesh-density/notebooks/mesh-simplification.ipynb) notebook. Here, we copy these back so we can test GLTF reconstruction from these. 

In [12]:
import pymeshlab

In [13]:
ms = pymeshlab.MeshSet()

# Loading the full mesh
ms.load_new_mesh('../../../models/foot/modelsoriginal_mesh.obj')

m = ms.current_mesh()
v_matrix_org = m.vertex_matrix()
f_matrix_org = m.face_matrix()

v_matrix_org

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.157481,  0.015595,  0.408668],
       [ 0.163158,  0.035477,  0.397872],
       [ 0.163247,  0.081139,  0.437404]], shape=(800, 3))

In [14]:
f_matrix_org

array([[366,  28,  30],
       [366,  30, 363],
       [ 81,  15,  16],
       ...,
       [395,  89, 489],
       [505, 490,  89],
       [490, 463,  89]], shape=(1586, 3), dtype=int32)

In [15]:
# Loading the decimated mesh
ms.load_new_mesh('../../../models/foot/modelsdecimated_mesh.obj')

m = ms.current_mesh()
v_matrix_dec = m.vertex_matrix()
f_matrix_dec = m.face_matrix()

v_matrix_dec

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.100379, -0.002407,  0.430997],
       [ 0.130661,  0.005967,  0.426889],
       [ 0.174218,  0.030867,  0.461069]], shape=(403, 3))

In [16]:
f_matrix_dec

array([[366,  28,  30],
       [366,  30, 363],
       [ 81,  15,  16],
       ...,
       [395, 357,  89],
       [395,  89, 390],
       [361, 231,  89]], shape=(792, 3), dtype=int32)

To drive this point home, the order of vertices in our decimated mesh should be exactly the same as in the original. Therefore, these can share the same vertex array, and only save different byteoffsets when writing to gltf. Let's test this out by calling random indices in both our arrays.

In [17]:
v_matrix_org[42]

array([ 0.025718, -0.011211,  0.498848])

In [18]:
v_matrix_dec[42]

array([ 0.025718, -0.011211,  0.498848])

We shall run a couple more assert statements.

In [19]:
assert v_matrix_org[32].all() == v_matrix_dec[32].all()
assert v_matrix_org[64].all() == v_matrix_dec[64].all()
assert v_matrix_org[128].all() == v_matrix_dec[128].all()
assert v_matrix_org[256].all() == v_matrix_dec[256].all()
assert v_matrix_org[399].all() == v_matrix_dec[399].all()

Perfect. We move on to converting these arrays to one GLTF file, making sure to substitute any duplicate arrays with their proper index. Before doing so, let's convert our arrays to a prespecified file type.

In [20]:
f_matrix_org = f_matrix_org.astype(np.uint32)
f_matrix_dec = f_matrix_dec.astype(np.uint32)

v_matrix_org = v_matrix_org.astype(np.float32)
v_matrix_dec = v_matrix_dec.astype(np.float32)

In [21]:
v_matrix_org

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.157481,  0.015595,  0.408668],
       [ 0.163158,  0.035477,  0.397872],
       [ 0.163247,  0.081139,  0.437404]], shape=(800, 3), dtype=float32)

In [22]:
triangles_org_binary_blob = f_matrix_org.flatten().tobytes()
triangles_dec_binary_blob = f_matrix_dec.flatten().tobytes()

points_org_binary_blob = v_matrix_org.tobytes()
points_dec_binary_blob = v_matrix_dec.tobytes()

# print(points_org_binary_blob, "\n", points_dec_binary_blob)

points_dec_byte_offset = v_matrix_dec.shape[0] # Here, instead of a binary blob for our decimated mesh, we subset from the original with a byte offset

gltf_lod = pygltflib.GLTF2(
    scene=0,
    scenes=[pygltflib.Scene(nodes=[0, 1])],
    nodes=[
        pygltflib.Node(mesh=0),
        pygltflib.Node(mesh=1)
    ],
    meshes=[
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=2), indices=0
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=3), indices=1
                )
            ]
        )
    ],
    accessors=[
        pygltflib.Accessor(  # accessor0: original mesh indices
            bufferView=0,
            componentType=5125,
            count=f_matrix_org.size,
            type=pygltflib.SCALAR
            # max=[int(f_matrix_org.max())],
            # min=[int(f_matrix_org.min())],
        ),
        pygltflib.Accessor(  # accessor1: decimated mesh indices
            bufferView=1,
            componentType=5125,
            count=f_matrix_dec.size,
            type=pygltflib.SCALAR
            # max=[int(f_matrix_dec.max())],
            # min=[int(f_matrix_dec.min())],
        ),
        pygltflib.Accessor(  # accessor2: original mesh vertex positions
            bufferView=2,
            componentType=pygltflib.FLOAT,
            count=len(v_matrix_org),
            type=pygltflib.VEC3,
            max=v_matrix_org.max(axis=0).tolist(),
            min=v_matrix_org.min(axis=0).tolist(),
        ),
        pygltflib.Accessor(  # accessor3: decimated mesh vertex positions
            bufferView=3,
            componentType=pygltflib.FLOAT,
            count=len(v_matrix_dec),
            type=pygltflib.VEC3,
            max=v_matrix_dec.max(axis=0).tolist(),
            min=v_matrix_dec.min(axis=0).tolist(),
        )
    ],
    bufferViews=[
        pygltflib.BufferView(  # bufferview0: original mesh indices
            buffer=0,
            byteLength=len(triangles_org_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER,
        ),
        pygltflib.BufferView(  # bufferView1: decimated mesh indices
            buffer=0,
            byteOffset=len(triangles_org_binary_blob),
            byteLength=len(triangles_dec_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER,
        ),
        pygltflib.BufferView(  # bufferView2: original mesh vertices
            buffer=0,
            byteOffset=len(triangles_org_binary_blob)+len(triangles_dec_binary_blob),
            byteLength=len(points_org_binary_blob),
            target=pygltflib.ARRAY_BUFFER,
        ),
        pygltflib.BufferView(  # bufferView3: decimated mesh vertices
            buffer=0,
            byteOffset=len(triangles_org_binary_blob)+len(triangles_dec_binary_blob), # Notice here how we are using the same starting index as the previous bufferview
            byteLength=len(points_dec_binary_blob),
            target=pygltflib.ARRAY_BUFFER,
        )
    ],
    buffers=[
        pygltflib.Buffer(
            byteLength=len(triangles_org_binary_blob) + len(triangles_dec_binary_blob) + len(points_org_binary_blob)
        )
    ],
)
gltf_lod.set_binary_blob(triangles_org_binary_blob + triangles_dec_binary_blob + points_org_binary_blob)

In [23]:
filename3 = "test_LOD.glb"
gltf_lod.save(filename3)

True

Now, let's see whats been saved.

![Output of manual GLTF conversion](../img/gltf-output-results.png)

This is perfect. We see that we have successfully converted our meshes into one glb file. Note the decimation being applied to the toes of the mesh. Note: These 2 meshes should share the same vertex coordinates, they have currently been adjusted to show the difference. To drive this pont home, lets open this same file in MeshLab, and compare to see if the vertex indices are the same.

Here, we color code our vertices - Red: vertices belonging to strictly decimated mesh, Blue: vertices belonging to both original and decimated mesh.

![Comparison of vertices- color and location](../img/gltf-output-results-vertex-alignment.png)

Now, let's compare vertex indices aross the mesh.

![Index alignment across both meshes](../img/gltf-output-results-vertex-index-match.png)

In the figure above, we zoom into the big toe of our mesh. We select a face in either mesh that appears to consist of the same vertices. In this example, the faces selected are composed of the same vertices - 171 and 175. Positionally, these indices are the same.

## What This Means

We have successfully proven that you can manually write the output of a GLTF file. We have also successfully proven that vertex arrays can be shared across objects in the scene, provided they share the same index structure. As a final proof of concept test, we manually convert our 2 created LOD meshes to GLTF file format in Blender. Since we aren't specifying explicitly that one is a subset of the other, we expect an additional vertex array for our decimated LOD mesh. We should expect to see a higher file size.

![Comparison of File Size- Manual vs Auto Export](../img/compariosn-of-file-size-manual-gltf.png)

Sure enough, we see that our "full" file (corresponding to the auto export) is larger than the manually exported file. As a reminder, both of these files consist of the same meshes. One was created automatically, by duplicating vertices, and one was created manually with careful consideration into the ordering of the vertices.

## Modularizing the Code

Now its time to modularize this code such that it can be called repeatedly for multiple objects in the scene. Here, we will work with the `piperacks` 3D model. This model was used for the [original LOD control case study](../../hosting-3d-model/per-object-lod-control-with-threejs.md) and is useful since it has multiple objects saved within it in a hierarchy format. The goal here is to intake this file, apply our decimation algorithm to each of the meshes, and reorder the file into our prescribed format. 

Let's start by loading in this model.

In [24]:
filename = "../../../models/piperack/piperacks_lod-100.glb"
gltf = pygltflib.GLTF2().load(filename)

We access the same attributes as before- the triangle and points array.

In [25]:
triangles_binary_blob = gltf.binary_blob()

triangles_accessor = gltf.accessors[gltf.meshes[0].primitives[0].indices]
triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
triangles = np.frombuffer(
    triangles_binary_blob[
        triangles_buffer_view.byteOffset
        + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
        + triangles_buffer_view.byteLength
    ],
    dtype="uint8",
    count=triangles_accessor.count,
).reshape((-1, 3))

triangles


array([[ 1,  0, 13],
       [ 0, 19,  0],
       [ 1,  0, 19],
       [ 0,  7,  0],
       [ 9,  0,  6],
       [ 0, 18,  0],
       [ 9,  0, 18],
       [ 0, 21,  0],
       [23,  0, 20],
       [ 0, 14,  0],
       [23,  0, 14],
       [ 0, 17,  0]], dtype=uint8)

In [26]:
points_binary_blob = gltf.binary_blob()

points_accessor = gltf.accessors[gltf.meshes[0].primitives[0].attributes.POSITION]
points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
points = np.frombuffer(
    points_binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype="float32",
    count=points_accessor.count * 3,
).reshape((-1, 3))

points

array([[ 1.,  1., -1.],
       [ 1.,  1., -1.],
       [ 1.,  1., -1.],
       [ 1., -1., -1.],
       [ 1., -1., -1.],
       [ 1., -1., -1.],
       [ 1.,  1.,  1.],
       [ 1.,  1.,  1.],
       [ 1.,  1.,  1.],
       [ 1., -1.,  1.],
       [ 1., -1.,  1.],
       [ 1., -1.,  1.],
       [-1.,  1., -1.],
       [-1.,  1., -1.],
       [-1.,  1., -1.],
       [-1., -1., -1.],
       [-1., -1., -1.],
       [-1., -1., -1.],
       [-1.,  1.,  1.],
       [-1.,  1.,  1.],
       [-1.,  1.,  1.],
       [-1., -1.,  1.],
       [-1., -1.,  1.],
       [-1., -1.,  1.]], dtype=float32)

The points array looks a little weird, but this is likely becuase the first item in the scene is the generic "cube" object that shows up by default. If we instead load mesh number 2, we see that there are indeed applicable coordinates in our scene.

In [27]:
points_binary_blob = gltf.binary_blob()

points_accessor = gltf.accessors[gltf.meshes[1].primitives[0].attributes.POSITION]
points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
points = np.frombuffer(
    points_binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype="float32",
    count=points_accessor.count * 3,
).reshape((-1, 3))

points

array([[ 1.8971880e+01,  1.9729252e+01, -1.0444579e-21],
       [ 1.8971880e+01,  1.9729252e+01, -1.0444579e-21],
       [ 1.8810524e+01,  2.0786034e+01,  4.0000458e-08],
       ...,
       [ 1.8407465e+01,  1.9682011e+01,  6.6889602e-01],
       [ 1.6805010e+01,  1.9682011e+01,  6.6889602e-01],
       [ 1.6788462e+01,  1.9573631e+01,  6.6889602e-01]],
      shape=(702, 3), dtype=float32)

For the purposes of this EDA notebook, we created a custom decimate script which has been saved to the `scripts/` folder.

In [28]:
import sys
import os

sys.path.append(os.path.abspath("../../.."))

from scripts.decimate import decimate_mesh

To confirm that this is working as intended, we shall run this script on a test mesh- in this case, the original foot model.

In [29]:
filename = "../../../models/foot/human-foot-hires.glb"
gltf = pygltflib.GLTF2().load(filename)

Here we convert the triangles in the mesh to array.

In [30]:
triangles_binary_blob = gltf.binary_blob()

triangles_accessor = gltf.accessors[gltf.meshes[0].primitives[0].indices]
triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
triangles_test = np.frombuffer(
    triangles_binary_blob[
        triangles_buffer_view.byteOffset
        + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
        + triangles_buffer_view.byteLength
    ],
    dtype="uint16",
    count=triangles_accessor.count,
).reshape((-1, 3))

triangles_test

array([[736,  31,  33],
       [736,  33, 733],
       [ 86,  16,  17],
       ...,
       [808,  97,  44],
       [660, 821,  96],
       [821, 665,  96]], shape=(1586, 3), dtype=uint16)

Same with the vertices in the mesh.

In [31]:
points_binary_blob = gltf.binary_blob()

points_accessor = gltf.accessors[gltf.meshes[0].primitives[0].attributes.POSITION]
points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
points_test = np.frombuffer(
    points_binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype="float32",
    count=points_accessor.count * 3,
).reshape((-1, 3))

points_test

array([[ 0.05960765,  0.38341936, -0.04792476],
       [-0.03059868, -0.01601027, -0.02000928],
       [ 0.05239549,  0.40234527,  0.10257383],
       ...,
       [ 0.17421806,  0.03086693,  0.4610686 ],
       [ 0.16315834,  0.03547674,  0.39787173],
       [-0.05680574,  0.01286671,  0.5708645 ]],
      shape=(822, 3), dtype=float32)

We see our familiar triangles and points arrays from earlier. Let's pass this into our function and see what we get.

We also create a simple cube object to pass into our function for testing purposes.

In [32]:
points = np.array(
    [
        [-0.5, -0.5, 0.5],
        [0.5, -0.5, 0.5],
        [-0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5],
        [0.5, -0.5, -0.5],
        [-0.5, -0.5, -0.5],
        [0.5, 0.5, -0.5],
        [-0.5, 0.5, -0.5],
    ],
    dtype="float32",
)
triangles = np.array(
    [
        [0, 1, 2],
        [3, 2, 1],
        [1, 0, 4],
        [5, 4, 0],
        [3, 1, 6],
        [4, 6, 1],
        [2, 3, 7],
        [6, 7, 3],
        [0, 2, 5],
        [7, 5, 2],
        [5, 7, 4],
        [6, 4, 7],
    ],
    dtype="uint8",
)
points

array([[-0.5, -0.5,  0.5],
       [ 0.5, -0.5,  0.5],
       [-0.5,  0.5,  0.5],
       [ 0.5,  0.5,  0.5],
       [ 0.5, -0.5, -0.5],
       [-0.5, -0.5, -0.5],
       [ 0.5,  0.5, -0.5],
       [-0.5,  0.5, -0.5]], dtype=float32)

In [33]:
points_rmp, triangles_rmp, point_dm, triangles_dm = decimate_mesh(points_test, triangles_test)

In [34]:
triangles_dm

array([[368,  28,  30],
       [368,  30, 365],
       [ 81,  15,  16],
       ...,
       [395, 357,  89],
       [395,  89, 392],
       [362, 229,  89]], shape=(792, 3), dtype=int32)

Seems to be working, however we note a key finding that the dtype of the array elements need to be accurately specified in order for it to work. Here, our cube object was able to load all the indices using `uint8` format since the max index in our cube is 8. However, in our foot model, the mesh index goes up until 800 hence we need a larger dtype- `uint16`.

For this reason, we define a GLTF mapping dictionary to ensure we correctly identify the dtype of each of our accessors.

In [35]:
# Code derived from Gemini
GLTF_COMPONENT_TYPES = {
    5120: np.int8,
    5121: np.uint8,
    5122: np.int16,
    5123: np.uint16,
    5125: np.uint32,
    5126: np.float32
}

And we can now tweak our existing triangles and points array extraction code as follows.

In [36]:
triangles_binary_blob = gltf.binary_blob()

triangles_accessor = gltf.accessors[gltf.meshes[0].primitives[0].indices]
triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
triangles_test = np.frombuffer(
    triangles_binary_blob[
        triangles_buffer_view.byteOffset
        + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
        + triangles_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType], # Changed this line to be dynamic
    count=triangles_accessor.count,
).reshape((-1, 3))

points_binary_blob = gltf.binary_blob()

points_accessor = gltf.accessors[gltf.meshes[0].primitives[0].attributes.POSITION]
points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
points_test = np.frombuffer(
    points_binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType], # Changed this line to be dynamic
    count=points_accessor.count * 3,
).reshape((-1, 3))


In [37]:
triangles_test

array([[736,  31,  33],
       [736,  33, 733],
       [ 86,  16,  17],
       ...,
       [808,  97,  44],
       [660, 821,  96],
       [821, 665,  96]], shape=(1586, 3), dtype=uint16)

In [38]:
points_test

array([[ 0.05960765,  0.38341936, -0.04792476],
       [-0.03059868, -0.01601027, -0.02000928],
       [ 0.05239549,  0.40234527,  0.10257383],
       ...,
       [ 0.17421806,  0.03086693,  0.4610686 ],
       [ 0.16315834,  0.03547674,  0.39787173],
       [-0.05680574,  0.01286671,  0.5708645 ]],
      shape=(822, 3), dtype=float32)

Looks good. This way we assign our array with the dtype that's been already saved to the gltf file.

The next step is to wrap this within a larger loop that will apply this function sequentially to every mesh in the gltf file.

First, let's load our piperacks model to the session.

In [39]:
filename = "../../../models/piperack/piperacks_lod-100.glb"
gltf = pygltflib.GLTF2().load(filename)

How many individual meshes do we have in this file?

In [40]:
len(gltf.meshes)

49

Loading this model into Blender, we can confirm that there are indeed 49 meshes (indicated by the upside down triangle icon beside each object)- 48 correspond to the piperack itself, while 1 is for the default Blender cube which probably got accidentally included as well.

![Piperacks model loaded to Blender confirms 49 meshes](../img/piperacks-high-lod-compare-total-objects.png)

Let's acquire the binary blobs that contain the raw data associated with our scene.

In [41]:
piperack_binary_blob = gltf.binary_blob()

Now, we proceed with writing our script. For now, we're focused on logic only and as a result, this code is not the most optimal version it could be. We start by initializing an empty container, and append the newly created decimated / remapped meshes to it.

In [53]:
# Helper functions for matrix transforms
def trs_to_matrix(t, r, s):
    """Converts Translation, Rotation (Quat), and Scale to a 4x4 Matrix."""
    # Translation matrix
    T = np.eye(4)
    if t: T[:3, 3] = t
    
    # Scale matrix
    S = np.eye(4)
    if s: np.fill_diagonal(S[:3, :3], s)
    
    # Rotation matrix from Quaternion [x, y, z, w]
    R = np.eye(4)
    if r:
        x, y, z, w = r
        R[:3, :3] = [
            [1 - 2*y**2 - 2*z**2, 2*x*y - 2*z*w,     2*x*z + 2*y*w],
            [2*x*y + 2*z*w,     1 - 2*x**2 - 2*z**2, 2*y*z - 2*x*w],
            [2*x*z - 2*y*w,     2*y*z + 2*x*w,     1 - 2*x**2 - 2*y**2]
        ]
    
    # glTF order: M = T * R * S
    return T @ R @ S

def compute_world_matrices(gltf):
    """Calculates absolute world matrices for all nodes in the gltf."""
    world_matrices = [np.eye(4) for _ in range(len(gltf.nodes))]
    
    # Find root nodes (nodes not listed as children of anyone)
    all_children = set()
    for node in gltf.nodes:
        if node.children:
            all_children.update(node.children)
    
    roots = [i for i in range(len(gltf.nodes)) if i not in all_children]

    def traverse(node_idx, parent_matrix):
        node = gltf.nodes[node_idx]
        if node.matrix:
            local_m = np.array(node.matrix).reshape(4, 4).T # Column-major to Row-major
        else:
            local_m = trs_to_matrix(node.translation, node.rotation, node.scale)
        
        world_m = parent_matrix @ local_m
        world_matrices[node_idx] = world_m
        
        if node.children:
            for child_idx in node.children:
                traverse(child_idx, world_m)

    for root_idx in roots:
        traverse(root_idx, np.eye(4))
        
    return world_matrices

In [57]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_lod = pygltflib.GLTF2()
gltf_lod.scenes.append(pygltflib.Scene(nodes=[]))
gltf_lod.scene = 0

# Empty array for appending binary blobs
main_binary_blob = bytearray()

# Compute world matrices for each mesh
world_matrices = compute_world_matrices(gltf)

# Variables for loop traversal
byte_offset_ctr = 0
bufferview_ctr = 0
accessor_ctr = 0

# Main loop
for node_idx, original_node in enumerate(gltf.nodes):    
    if original_node.mesh is None:
        continue
    
    mesh_idx = original_node.mesh
    primitive = gltf.meshes[mesh_idx].primitives[0]

    # Step 1: Acquire the existing triangles and points array from the gltf file
    triangles_accessor = gltf.accessors[primitive.indices]
    triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
    triangles = np.frombuffer(
        piperack_binary_blob[
            triangles_buffer_view.byteOffset
            + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
            + triangles_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
        count=triangles_accessor.count,
    ).reshape((-1, 3))

    points_accessor = gltf.accessors[primitive.attributes.POSITION]
    points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
    points = np.frombuffer(
        piperack_binary_blob[
            points_buffer_view.byteOffset
            + points_accessor.byteOffset : points_buffer_view.byteOffset
            + points_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
        count=points_accessor.count * 3,
    ).reshape((-1, 3))

    # Step 2: Apply the decimation and remapping function
    points_rmp, triangles_rmp, points_dm, triangles_dm = decimate_mesh(points, triangles)

    triangles_rmp = triangles_rmp.astype(np.uint32)
    triangles_dm = triangles_dm.astype(np.uint32)

    points_rmp = points_rmp.astype(np.float32)
    points_dm = points_dm.astype(np.float32)

    # Step 3: Convert the new arrays to binary and append to main binary blob
    triangles_org_binary_blob = triangles_rmp.flatten().tobytes()
    triangles_dec_binary_blob = triangles_dm.flatten().tobytes()

    points_org_binary_blob = points_rmp.tobytes()
    points_dec_binary_blob = points_dm.tobytes()

    combined_byte_array = triangles_org_binary_blob + triangles_dec_binary_blob + points_org_binary_blob
    main_binary_blob.extend(combined_byte_array) # Note we do not append the decimated mesh vertices

    # Step 4: Format the GLTF file with the 2 new LODs
    # Append the BufferViews
    gltf_lod.bufferViews.extend([
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr, 
            byteLength=len(triangles_org_binary_blob), 
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr + len(triangles_org_binary_blob), 
            byteLength=len(triangles_dec_binary_blob), 
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr + len(triangles_org_binary_blob) + len(triangles_dec_binary_blob), 
            byteLength=len(points_org_binary_blob), 
            target=pygltflib.ARRAY_BUFFER
        ),
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr + len(triangles_org_binary_blob) + len(triangles_dec_binary_blob), 
            byteLength=len(points_dec_binary_blob), 
            target=pygltflib.ARRAY_BUFFER
        )
    ])

    # Append the Accessors
    gltf_lod.accessors.extend([
        pygltflib.Accessor(
            bufferView=bufferview_ctr,
            componentType=5125,
            count=triangles_rmp.size,
            type=pygltflib.SCALAR
        ),
        pygltflib.Accessor(
            bufferView=bufferview_ctr+1,
            componentType=5125,
            count=triangles_dm.size,
            type=pygltflib.SCALAR
        ),
        pygltflib.Accessor(
            bufferView=bufferview_ctr+2,
            componentType=pygltflib.FLOAT,
            count=len(points_rmp),
            type=pygltflib.VEC3,
            max=points_rmp.max(axis=0).tolist(),
            min=points_rmp.min(axis=0).tolist(),
        ),
        pygltflib.Accessor(
            bufferView=bufferview_ctr+3,
            componentType=pygltflib.FLOAT,
            count=len(points_dm),
            type=pygltflib.VEC3,
            max=points_dm.max(axis=0).tolist(),
            min=points_dm.min(axis=0).tolist(),
        )
    ])

    # Append the Meshes
    gltf_lod.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=bufferview_ctr+2), indices=bufferview_ctr
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=bufferview_ctr+3), indices=bufferview_ctr+1
                )
            ]
        )
    ])

    world_m = world_matrices[node_idx]
    flat_m = world_m.T.flatten().tolist()

    # Append the Nodes
    gltf_lod.nodes.extend([
        pygltflib.Node(
            mesh=accessor_ctr,
            matrix=flat_m
        ),
        pygltflib.Node(
            mesh=accessor_ctr+1,
            matrix=flat_m
        )
    ])

    # Add to the existing scene
    gltf_lod.scenes[0].nodes.extend([accessor_ctr, accessor_ctr+1])


    # Step 6: Manually update the counters
    byte_offset_ctr = byte_offset_ctr + len(triangles_org_binary_blob) + len(triangles_dec_binary_blob) + len(points_org_binary_blob)
    bufferview_ctr = bufferview_ctr + 4
    accessor_ctr = accessor_ctr + 2

gltf_lod.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
gltf_lod.set_binary_blob(main_binary_blob)

filename4 = "test_piperacks_LOD.glb"
gltf_lod.save(filename4)


True

After a fair bit of trial and error, we now have a working script. Loading this newly created GLTF file into blender, this is what we see.

![Initial Load of the Recreated GLTF file](../img/gltf-remapped-initial%20load.png)

Position, rotation and scale appears to have translated correctly. What about the LOD's?

![Comparison of HighRes and LowRes LODs in GLTF file](../img/gltf-remapped-comparison.png)

The decimation appears to be very high however, we do see that the vertex number has been successfully compressed.

As for the memory- we observe that this new file is 6.9 MB file size in total. While we likely have not included any materials, metadata and other elements, this seems to be a good memory size reduction. To confirm, we can tweak our above code a touch to also include the decimated mesh vertices when creating the new GLTF file.

In [56]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_lod = pygltflib.GLTF2()
gltf_lod.scenes.append(pygltflib.Scene(nodes=[]))
gltf_lod.scene = 0

# Empty array for appending binary blobs
main_binary_blob = bytearray()

# Compute world matrices for each mesh
world_matrices = compute_world_matrices(gltf)

# Variables for loop traversal
byte_offset_ctr = 0
bufferview_ctr = 0
accessor_ctr = 0

# Main loop
for node_idx, original_node in enumerate(gltf.nodes):    
    if original_node.mesh is None:
        continue
    
    mesh_idx = original_node.mesh
    primitive = gltf.meshes[mesh_idx].primitives[0]

    # Step 1: Acquire the existing triangles and points array from the gltf file
    triangles_accessor = gltf.accessors[primitive.indices]
    triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
    triangles = np.frombuffer(
        piperack_binary_blob[
            triangles_buffer_view.byteOffset
            + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
            + triangles_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
        count=triangles_accessor.count,
    ).reshape((-1, 3))

    points_accessor = gltf.accessors[primitive.attributes.POSITION]
    points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
    points = np.frombuffer(
        piperack_binary_blob[
            points_buffer_view.byteOffset
            + points_accessor.byteOffset : points_buffer_view.byteOffset
            + points_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
        count=points_accessor.count * 3,
    ).reshape((-1, 3))

    # Step 2: Apply the decimation and remapping function
    points_rmp, triangles_rmp, points_dm, triangles_dm = decimate_mesh(points, triangles)

    triangles_rmp = triangles_rmp.astype(np.uint32)
    triangles_dm = triangles_dm.astype(np.uint32)

    points_rmp = points_rmp.astype(np.float32)
    points_dm = points_dm.astype(np.float32)

    # Step 3: Convert the new arrays to binary and append to main binary blob
    triangles_org_binary_blob = triangles_rmp.flatten().tobytes()
    triangles_dec_binary_blob = triangles_dm.flatten().tobytes()

    points_org_binary_blob = points_rmp.tobytes()
    points_dec_binary_blob = points_dm.tobytes()

    combined_byte_array = triangles_org_binary_blob + triangles_dec_binary_blob + points_org_binary_blob + points_dec_binary_blob
    main_binary_blob.extend(combined_byte_array) # Note we do not append the decimated mesh vertices

    # Step 4: Format the GLTF file with the 2 new LODs
    # Append the BufferViews
    gltf_lod.bufferViews.extend([
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr, 
            byteLength=len(triangles_org_binary_blob), 
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr + len(triangles_org_binary_blob), 
            byteLength=len(triangles_dec_binary_blob), 
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr + len(triangles_org_binary_blob) + len(triangles_dec_binary_blob), 
            byteLength=len(points_org_binary_blob), 
            target=pygltflib.ARRAY_BUFFER
        ),
        pygltflib.BufferView(
            buffer=0, 
            byteOffset=byte_offset_ctr + len(triangles_org_binary_blob) + len(triangles_dec_binary_blob) + len(points_org_binary_blob), 
            byteLength=len(points_dec_binary_blob), 
            target=pygltflib.ARRAY_BUFFER
        )
    ])

    # Append the Accessors
    gltf_lod.accessors.extend([
        pygltflib.Accessor(
            bufferView=bufferview_ctr,
            componentType=5125,
            count=triangles_rmp.size,
            type=pygltflib.SCALAR
        ),
        pygltflib.Accessor(
            bufferView=bufferview_ctr+1,
            componentType=5125,
            count=triangles_dm.size,
            type=pygltflib.SCALAR
        ),
        pygltflib.Accessor(
            bufferView=bufferview_ctr+2,
            componentType=pygltflib.FLOAT,
            count=len(points_rmp),
            type=pygltflib.VEC3,
            max=points_rmp.max(axis=0).tolist(),
            min=points_rmp.min(axis=0).tolist(),
        ),
        pygltflib.Accessor(
            bufferView=bufferview_ctr+3,
            componentType=pygltflib.FLOAT,
            count=len(points_dm),
            type=pygltflib.VEC3,
            max=points_dm.max(axis=0).tolist(),
            min=points_dm.min(axis=0).tolist(),
        )
    ])

    # Append the Meshes
    gltf_lod.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=bufferview_ctr+2), indices=bufferview_ctr
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(POSITION=bufferview_ctr+3), indices=bufferview_ctr+1
                )
            ]
        )
    ])

    world_m = world_matrices[node_idx]
    flat_m = world_m.T.flatten().tolist()

    # Append the Nodes
    gltf_lod.nodes.extend([
        pygltflib.Node(
            mesh=accessor_ctr,
            matrix=flat_m
        ),
        pygltflib.Node(
            mesh=accessor_ctr+1,
            matrix=flat_m
        )
    ])

    # Add to the existing scene
    gltf_lod.scenes[0].nodes.extend([accessor_ctr, accessor_ctr+1])


    # Step 6: Manually update the counters
    byte_offset_ctr = byte_offset_ctr + len(triangles_org_binary_blob) + len(triangles_dec_binary_blob) + len(points_org_binary_blob) + len(points_dec_binary_blob)
    bufferview_ctr = bufferview_ctr + 4
    accessor_ctr = accessor_ctr + 2

gltf_lod.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
gltf_lod.set_binary_blob(main_binary_blob)

filename4 = "test_piperacks_LOD_2.glb"
gltf_lod.save(filename4)

True

Unfortunately, looks like the file size out of this new script is exactly the same. For some reason, the quality of the decimation appears to be better as well.

![Comparison of Discrete v Reused Vertex Arrays in GLTF file](../img/gltf-remapped-discrete-v-reused.png)

Observe how the reused vertex arrays have poorer decimation than the discrete one.

We will need to explore this more later. For now, it seems like the script is working as intended.